# Vector Search via MCP

This notebook demonstrates **semantic vector search** against a Neo4j knowledge graph using the **Model Context Protocol (MCP)**. Instead of connecting to Neo4j directly with a driver, an AI agent uses MCP tools to query the database.

**Learning Objectives:**
- Connect to a Neo4j MCP server through AgentCore Gateway
- Generate query embeddings using Amazon Bedrock
- Perform vector similarity search through MCP tools
- Compare semantic search results at different `top_k` values

**How it works:**
1. The notebook embeds the user's question using Bedrock Titan embeddings
2. The agent receives the embedding and uses MCP's `execute-query` tool to run vector search Cypher
3. Neo4j's vector index returns the most semantically similar chunks
4. The agent formats and presents the results

**Prerequisites:**
- `../CONFIG.txt` configured with `MODEL_ID`, `REGION`, `MCP_GATEWAY_URL`, and `MCP_ACCESS_TOKEN`
- Data loaded into Neo4j with vector embeddings on Chunk nodes (from earlier labs)

In [ ]:
%pip install -U langgraph langchain-aws langchain-mcp-adapters mcp nest-asyncio -q

## 1. Configuration and Imports

In [ ]:
import json
import nest_asyncio
from dotenv import load_dotenv
import os

import boto3
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

nest_asyncio.apply()

# Load configuration
load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
EMBEDDING_MODEL_ID = os.getenv("EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

# Derive BASE_MODEL_ID for cross-region inference profiles
if MODEL_ID and MODEL_ID.startswith("us.anthropic."):
    BASE_MODEL_ID = MODEL_ID.replace("us.anthropic.", "anthropic.")
elif MODEL_ID and MODEL_ID.startswith("global.anthropic."):
    BASE_MODEL_ID = MODEL_ID.replace("global.anthropic.", "anthropic.")
else:
    BASE_MODEL_ID = None

print(f"Model:     {MODEL_ID}")
print(f"Embedder:  {EMBEDDING_MODEL_ID}")
print(f"Region:    {REGION}")
print(f"Gateway:   {MCP_GATEWAY_URL[:50]}..." if MCP_GATEWAY_URL and len(MCP_GATEWAY_URL) > 50 else f"Gateway:   {MCP_GATEWAY_URL}")

# Validate MCP config
assert MCP_GATEWAY_URL and MCP_GATEWAY_URL != "your-gateway-url-here", \
    "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN and MCP_ACCESS_TOKEN != "your-access-token-here", \
    "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print("\nConfiguration loaded!")

## 2. Embedding Helper

To perform vector search, we need to convert the user's question into an embedding vector. This function calls Amazon Bedrock's Titan embedding model to generate a 1024-dimensional vector that captures the semantic meaning of the text.

The agent will use this embedding in a Cypher query against Neo4j's `chunkEmbeddings` vector index.

In [ ]:
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)


def get_embedding(text: str) -> list[float]:
    """Generate an embedding vector for the given text using Bedrock Titan."""
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),
    )
    result = json.loads(response["body"].read())
    return result["embedding"]


# Test the embedding function
test_embedding = get_embedding("What are Apple's main products?")
print(f"Embedding dimensions: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

## 3. Connect to MCP Server and Initialize Agent

Connect to the Neo4j MCP server through the AgentCore Gateway. The `MultiServerMCPClient` discovers the available tools (typically `get-schema` and `execute-query`) and makes them available to the LangGraph agent.

The system prompt instructs the agent to use vector search via the `db.index.vector.queryNodes` Cypher procedure, which finds chunks whose embeddings are most similar to the query embedding.

In [ ]:
VECTOR_SEARCH_PROMPT = """You are a retrieval assistant that performs semantic vector search against a Neo4j database containing SEC 10-K filing data.

You have access to MCP tools that let you execute Cypher queries against the database.

VECTOR SEARCH INSTRUCTIONS:
When given a query embedding (a list of floats), use this Cypher pattern to find semantically similar text chunks:

CALL db.index.vector.queryNodes('chunkEmbeddings', $top_k, $embedding)
YIELD node, score
RETURN node.text AS text, score
ORDER BY score DESC

- The vector index is named 'chunkEmbeddings' and is on Chunk nodes
- The embedding will be provided in the user's message as a JSON array
- Use the exact embedding provided — do not modify it
- Return the chunk text and similarity score
- Always ORDER BY score DESC

FORMAT:
For each result, show:
1. The similarity score (higher = more similar)
2. A preview of the chunk text (first 200 characters)
"""

# Initialize LLM
llm_kwargs = {
    "model": MODEL_ID,
    "region_name": REGION,
    "temperature": 0,
}
if BASE_MODEL_ID:
    llm_kwargs["base_model_id"] = BASE_MODEL_ID

llm = ChatBedrockConverse(**llm_kwargs)

# Connect to MCP server
mcp_client = MultiServerMCPClient(
    {
        "neo4j": {
            "transport": "streamable_http",
            "url": MCP_GATEWAY_URL,
            "headers": {
                "Authorization": f"Bearer {MCP_ACCESS_TOKEN}",
            },
        }
    }
)

tools = await mcp_client.get_tools()
print(f"MCP tools discovered: {[t.name for t in tools]}")

# Create the agent
agent = create_react_agent(llm, tools, prompt=VECTOR_SEARCH_PROMPT)
print("Agent ready!")

## 4. Vector Search Helper

The `vector_search` function handles the full retrieval pipeline:
1. Embed the query text using Bedrock Titan
2. Send the embedding to the agent
3. The agent calls the MCP `execute-query` tool with a vector search Cypher query
4. Return the agent's formatted results

This mirrors what `VectorRetriever` from neo4j-graphrag does internally, but the query execution goes through MCP instead of a direct Neo4j driver connection.

In [ ]:
async def vector_search(query: str, top_k: int = 5):
    """Embed a query and run vector search through the MCP agent."""
    print(f'Query: "{query}"')
    print(f"Top K: {top_k}")
    print("-" * 60)

    # Generate embedding for the query
    embedding = get_embedding(query)

    # Build the message with the embedding for the agent
    message = (
        f"Run a vector search for the following query. Use top_k={top_k}.\n\n"
        f"Query: {query}\n\n"
        f"Embedding (use this exact array in the Cypher query):\n{json.dumps(embedding)}"
    )

    result = await agent.ainvoke({"messages": [HumanMessage(content=message)]})

    response = result["messages"][-1].content
    print(f"\n{response}")
    return result

## 5. Run Vector Searches

Search for semantically similar chunks. The vector index finds chunks whose embeddings are closest to the query embedding, even when the exact words differ from the source text.

In [ ]:
# Search for product-related information
result = await vector_search("What are Apple's main products?", top_k=5)

In [ ]:
# Search for risk factor information
result = await vector_search("What are the key risk factors mentioned in SEC filings?", top_k=5)

In [ ]:
# Compare top_k values — fewer results for a focused search
result = await vector_search("What financial metrics indicate company performance?", top_k=3)

### A Note on Embedding Handling

The current approach serializes the full 1024-dimensional embedding (~8-10KB of JSON) into the agent's message. The agent then copies this numeric array into the Cypher query tool call. This works but is inefficient — it consumes context window tokens with data the LLM cannot meaningfully interpret, and risks the LLM truncating or modifying the array.

In [notebook 03](03_fulltext_hybrid_search_mcp.ipynb), you will see a cleaner approach: wrapping embedding generation and MCP query execution inside a custom `@tool` function. The agent calls `vector_search("Apple products")` instead of receiving a giant float array.

## Summary

You performed semantic vector search against a Neo4j knowledge graph through MCP:

| Component | Role |
|-----------|------|
| **Bedrock Titan** | Embeds the query text into a 1024-dimensional vector |
| **MCP Agent** | Receives the embedding and constructs a vector search Cypher query |
| **MCP execute-query** | Runs the Cypher against Neo4j via AgentCore Gateway |
| **Neo4j Vector Index** | Returns chunks ranked by cosine similarity |

The vector search returns isolated chunks — the most semantically similar text fragments. It doesn't know about the chunk's document, its neighbors, or related entities in the graph.

In the next notebook, we add **graph traversal** to pull in surrounding context alongside each vector match.

---

**Next:** [Graph-Enriched Search via MCP](02_graph_enriched_search_mcp.ipynb)